## Build a Proper ML Pipeline with Feature Engineering

In [68]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report
import joblib

import warnings
warnings.filterwarnings("ignore")

## 1. Loading Data

In [69]:
churn_data = pd.read_csv("../data/telco_churn.csv")
churn_data

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,6840-RESVB,Male,0,Yes,Yes,24,Yes,Yes,DSL,Yes,...,Yes,Yes,Yes,Yes,One year,Yes,Mailed check,84.80,1990.5,No
7039,2234-XADUH,Female,0,Yes,Yes,72,Yes,Yes,Fiber optic,No,...,Yes,No,Yes,Yes,One year,Yes,Credit card (automatic),103.20,7362.9,No
7040,4801-JZAZL,Female,0,Yes,Yes,11,No,No phone service,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.60,346.45,No
7041,8361-LTMKD,Male,1,Yes,No,4,Yes,Yes,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Mailed check,74.40,306.6,Yes


In [70]:
churn_data['TotalCharges'] = pd.to_numeric(churn_data['TotalCharges'], errors='coerce')
churn_data = churn_data.dropna(subset=['TotalCharges'])
churn_data = churn_data.drop(columns=['customerID'])
churn_data['Churn'] = churn_data['Churn'].map({'Yes': 1, 'No': 0})

## 2. Feature Engineering

In [71]:
churn_data['Avg_Monthly_Spend'] = churn_data['TotalCharges'] / churn_data['tenure'].replace(0, 1)

service_cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
churn_data['Num_Services'] = churn_data[service_cols].apply(lambda row: (row == 'Yes').sum(), axis=1)

#### Define X, y, and Column Groups

In [72]:
X = churn_data.drop(columns=['Churn'])
y = churn_data['Churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include='object').columns.tolist()

## 3. Building the ML pipeline

In [73]:
preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), categorical_features)
])

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(max_iter=1000))
])


## 4. Train, Predict, Evaluate

In [74]:
pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)
y_prob = pipeline.predict_proba(X_test)[:, 1]

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"F1:       {f1_score(y_test, y_pred):.4f}")
print(f"ROC-AUC:  {roc_auc_score(y_test, y_prob):.4f}")
print(classification_report(y_test, y_pred, target_names=['No Churn', 'Churn']))

Accuracy: 0.8031
F1:       0.6049
ROC-AUC:  0.8361
              precision    recall  f1-score   support

    No Churn       0.85      0.89      0.87      1033
       Churn       0.65      0.57      0.60       374

    accuracy                           0.80      1407
   macro avg       0.75      0.73      0.74      1407
weighted avg       0.80      0.80      0.80      1407



#### Compare to Manual Baseline

In [75]:
print("=== TASK 1 MANUAL BASELINE ===")
print("Accuracy: 0.8045 | F1: 0.6088 | ROC-AUC: 0.8361")
print("=== TASK 2 PIPELINE RESULT ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f} | F1: {f1_score(y_test, y_pred):.4f} | ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}")

=== TASK 1 MANUAL BASELINE ===
Accuracy: 0.8045 | F1: 0.6088 | ROC-AUC: 0.8361
=== TASK 2 PIPELINE RESULT ===
Accuracy: 0.8031 | F1: 0.6049 | ROC-AUC: 0.8361


## 5. Save the Pipeline

In [76]:
joblib.dump(pipeline, 'churn_pipeline.joblib')
print("Pipeline saved!")

Pipeline saved!


## Overall Summary
- The pipeline achieved virtually identical performance to the manual approach 
(Accuracy: 0.8031 vs 0.8045, F1: 0.6049 vs 0.6088, ROC-AUC: 0.8361 for both — 
matching exactly). The tiny difference (under 0.4%) comes from minor 
implementation details between `OneHotEncoder` and the manual encoding approach 
used earlier, not from any loss of information or a flawed pipeline.

- This confirms the pipeline is a safe, reliable replacement for manual 
preprocessing — with the added benefit that scaling and encoding are now learned 
only from the training set and applied consistently to new data, preventing 
data leakage. This is also what makes the pipeline deployment-ready: it can be 
saved as a single object (`churn_pipeline.joblib`) and loaded directly by a 
future Streamlit app to make predictions on raw user input, without rewriting 
any preprocessing code.